# Modelado en Optimización (IIND-2501)

## Lección 3.5: Conceptos básicos de dualidad en programación lineal



Todo problema de Programación Lineal (PL) tiene asociado un **problema dual** con propiedades útiles. La dualidad nos da:
- **Interpretación económica:** el “valor” marginal de los recursos (precios sombra).
- **Cotas matemáticas:** límites superiores/inferiores sobre el valor óptimo.
- **Base algorítmica:** métodos como *Simplex dual*, condiciones de optimalidad.

In [1]:
# Imports y helpers
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
import sys
sys.path.append(str(Path.cwd() / "src"))

%load_ext autoreload
%autoreload 2

from helpers_dual import (
    plot_primal_1d_continuous,
    plot_primal_1var_multiconst,
    solve_primal_pulp,
    pretty_dual_generic,
    check_complementary_slackness,
)
plt.rcParams['figure.figsize'] = (7,4)

ImportError: cannot import name 'plot_primal_1d_continuous' from 'helpers_dual' (C:\Users\ch.gomez171\Documents\GitHub\Modelado-en-Optimizacion\helpers_dual.py)

## 1) Ejemplo ilustrativo: *Limonada'e Mango*

*Limonada'e Mango* es el emprendimiento de un estudiante que **suple litros de jugo** a vendedores de puestos en la ciclovía bogotana.

- **Ingredientes**
  - Concentrado de **limón** $\alpha_L$ litros por litro de producto.
  - Pulpa de **mango** $\alpha_M$ litros por litro de producto.
  - El resto es agua endulzada (sin costo limitante en el modelo).
- **Disponibilidades diarias**: $b_L$ litros de limón, $b_M$ litros de mango.
- **Precio/beneficio** por litro vendido: $p$.

**Problema Primal:** variable de decisión, $x$, litros a producir/vender (continuo). 
$$\max\ z = p\,x$$
$$s.a.$$
$$\alpha_L\,x \le b_L$$
$$\alpha_M\,x \le b_M$$
$$x \ge 0,$$

### Versión 1: Un recurso
Para iniciar, trabajaremos solo con **un recurso** (concentrado de limón), y luego con más recursos para consolidar la intuición.

$$\max\ z = 4 x$$
$$s.a.$$
$$0.5 \,x \le 10$$
$$x \ge 0,$$

In [ ]:
p = 4.0
alpha_L = 0.5   # 0.5 L de limón por litro de producto
b_L = 12.0      # 10 L disponibles de limón

plot_primal_1d_continuous(gain_per_liter=p, alpha=alpha_L, resource_available=b_L,
                          resource_label='Limón')

#### Significa que:
- Con el recurso actual (12 litros limón), **podemos producir hasta 24 litros** (de limonada), con ganancia $z=\$4(24)=\$96$
- Si tuviéramos **1 litro más de recurso**, podríamos producir **2 litros más de limonada**, con ganancia extra de $\$4(2)=\$8$
- Así que estaríamos dispuestos a pagar **hasta** $\$8$ por un litro adicional de concentrado de limón.

### Versión 2: Dos recursos

**Problema primal:**
$$\max\ z = 4x$$
$$\text{s.a.}$$
$$0.4 x \le 12,\quad \text{(limón)}$$
$$0.4 x \le 10,\quad \text{(mango)}$$
$$x\ge 0.$$


In [ ]:
alpha_L, alpha_M = 0.5, 0.5
B_L, B_M = 12.0, 10.0
p = 4.0
plot_primal_1var_multiconst(
    gain_per_liter=p,
    alphas=[alpha_L, alpha_M],
    bs=[B_L, B_M],
    labels=['Límite Limón', 'Límite Mango']
)

### Significa que:
- Podemos producir máximo 25 litros de limonada, agotando los 10 litros de mango.
- 

Planteamos un problema para encontrar el **"precio adecuado" por litro de limón**. Definimos $w_L$ como la variable (incógnita) que representa dicho precio.

$$\min\ b_L\,w_L$$
$$\text{s.a.}$$
$$\alpha_L\,w_L \ge p$$
$$w_L \ge 0$$
Interpretación: el proveedor/mercado del limón fija un precio mínimo por litro tal que al productor no le convenga vender por debajo de \(p\).